## 本章为langchain的基础应用，为本人自己总结且可以运行的代码，用于学习langchain

### 1.安装依赖（本文用的包管理工具为uv，若使用pip的同学需可以根据pyproject.toml安装依赖）
langchain：`uv add langchain`

langchain community：`uv add langchain_community`


### 2.apikey管理
这里我们使用到了dotenv来管理apikey，避免在代码中直接暴露apikey

In [ ]:
from platform import system

from langchain_core.messages import SystemMessage

# 创建.env文件，用于存储apikey等信息，这里我配置了Base_url和模型的名称
# LLM 配置
API_KEY=xxxx
BASE_URL=xxxx
MODEL_NAME=xxxx

In [ ]:
# 从.env文件中加载apikey等信息
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
MODEL_NAME = os.getenv("MODEL_NAME")

### 3.创建llm与交互
这里我们使用到了OpenAI来创建llm，也可以使用其他llm，如deepseek等，但是要注意这里的base_url一定要是openai版本的，不能写成anthropic版本的

In [ ]:
from langchain_openai import ChatOpenAI

# 使用OpenAI的客户端创建llm对象
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model_name=MODEL_NAME,
)

# 调用模型
response = llm.invoke("你好")
print(response.content)

In [ ]:
# 这里我们可以看一下response的完整结构
print(response.model_dump())

In [ ]:
# 因此我们可以在调试的时候打印出重要的语句
print(f"""
===== AI Response =====
模型: {response.response_metadata["model_name"]}
输入Token: {response.usage_metadata["input_tokens"]}
输出Token: {response.usage_metadata["output_tokens"]}
推理Token: {response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)}
结束原因: {response.response_metadata["finish_reason"]}

回复内容:
{response.content}
=======================
""")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

"""
我们常见的消息类型一共有四种
1. HumanMessage：用户消息
2. AIMessage：模型回复
3. SystemMessage：系统消息
4. ToolMessage：工具调用消息
由于我们需要与模型交互，所以需要使用HumanMessage来发送用户消息，模型回复则使用AIMessage来接收，而系统消息则使用SystemMessage来设置，工具调用消息则使用ToolMessage来设置模型的工具调用
"""
system_prompt = SystemMessage(content="你是一个专业的翻译，你的任务是将用户输入的中文翻译成英文")
human_prompt = HumanMessage(content="你好，我的名字叫约翰")

llm.invoke([system_prompt, human_prompt])


In [ ]:
# 下面是流式输出，通过每一个thunk进行输出
stream = llm.stream([system_prompt, human_prompt])
for thunk in stream:
    print(thunk.text, end="")


### 4. LangChain Expression Language
其实就是通过 | 来连接上下文，实现链式调用，传递输出结果给下一个组件
其实langchain的本质就是声明组件+连接组件

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 创建一个字符串输出解析器,本质就是获取模型回复的content
parser = StrOutputParser()

# 由于chain的调用需要用到Runnable对象，这里把prompt也转换为Runnable对象
prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    human_prompt,
])

# 构造链
chain = prompt | llm | parser

message = chain.invoke({})
print(message)

### 5.保存历史记录
在LangChain中提供了一个BaseChatMessageHistory的父类，用于保存历史记录，默认情况下LangChain只提供了一个基于内存的InMemoryChatMessageHistory来保存聊天记录

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

# 第一轮对话
history.add_user_message("1+1等于多少")
ai_message = llm.invoke(history.messages)
history.add_ai_message(ai_message | parser)

# 第二轮对话
history.add_user_message("那再+1呢")
fin_message = llm.invoke(history.messages)
print(fin_message.content)

也可以使用redis对历史记录进行保存，你需要安装 `uv add -q langchain-redis redis`，也可以用其他的数据库进行保存，可以在官网上查看

### 6.LangChain中的tools
LangChain中的tools机制，可以理解成一句话：Tool就是把函数包装成“模型能看懂、能决定是否调用、能按结构传参”的外部能力。
用人话就是说：大模型给函数参数，函数黑盒执行任务，返回结果给大模型，融合历史会话，函数结果，生成本次的回复，决定是否继续调用还是停止，输出结果。
这里我解释一下模型怎么知道需要调用什么函数，函数要提供哪些参数：
* 在拿到函数的时候，会通过docstring机制来获取注释中的函数的意思，那么就是写道越清晰，模型越能理解函数的意思
* 当bind_tools的时候，LangChain会偷偷把它转换成一个工具描述(schema)给模型，大致如下：{
  "name": "get_weather",
  "description": "查询某个城市的天气",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string"
      }
    },
    "required": ["city"]
  }
}
* 模型会根据schema来判断是否需要调用这个函数，以及调用的参数
* 那么就很明显了，函数的参数如果写的越详细，模型才能理解需要传递哪些参数，比如有个要传入订单id的函数，当你传入id，大模型可能不知道这个id是什么意思，如果写成order_id，那么大模型就会更好的理解这个函数需要传递的参数是什么，那么就能更好的理解这个函数是什么意思了。

In [ ]:
from langchain_core.tools import tool

# 定义一个工具函数，用于计算两个整数的和
@tool
def add(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b

# 绑定工具函数到模型中
llm_with_tools = llm.bind_tools([add])

# 调用函数
response = llm_with_tools.invoke("1+1等于多少")

print(response.tool_calls)
#你可以看到输出的结果有name，参数，id，type

上述内容拆解的结果如下（手动调用tool）

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [HumanMessage("1+1 等于多少？")]

ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    if tool_call["name"] == "add":
        result = add.invoke(tool_call["args"])

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

final_msg = llm_with_tools.invoke(messages)
print(final_msg.content)

"""
流程：
用户问题
→ 模型生成 tool_calls
→ 代码执行工具
→ 把工具结果用 ToolMessage 塞回 messages
→ 模型根据工具结果生成最终回答
"""

#### 6.1使用langchain的create_agent机制来实现一个简单智能体

In [ ]:
from langgraph.prebuilt import create_react_agent

# 最新版本用的是create_agent，我这里不知道为什么出现了问题，无法使用，不过企业级agent一般用langgraph来构建，这里不多解释
agent = create_react_agent(
    model=llm,
    tools=[add],
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "3 加 5 等于多少？"}
    ]
})

print(result)